# Listen, Attend and Spell — attention-based encoder-decoder for speech (2015)

_Papers · Speech & Audio_

**A pyramidal BiLSTM 'Listener' shrinks the audio in time; an attention 'Speller' decodes characters one at a time — one joint, end-to-end speech recognizer.**

---

This notebook is a practice scaffold for the **Listen, Attend and Spell — attention-based encoder-decoder for speech (2015)** lesson. We assemble LAS one component at a time — the pyramid downsampling, content attention, the Listener, the Speller — then train it on a toy task and read its alignment. Run the cells, read the note above each, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Two worked examples: pyramid downsampling and one attention step

Two small hand-checkable pieces anchor the whole model. **(a)** The pyramid (Eqn 5) concatenates each adjacent pair of time steps into one wider step, halving the sequence length each layer — so $8 \to 4 \to 2 \to 1$ over three layers. **(b)** One content-attention step (Eqns 9–11) scores each encoder feature against the decoder state, softmaxes the scores into weights $\alpha$, and returns their weighted sum as the context $c$.

With $\phi=\psi=$ identity the score is just a dot product, so you can verify $\alpha=[0.5065, 0.1863, 0.3072]$ and $c=[1.320, 0.680]$ by hand.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Worked example (a): pyramid time-downsampling, Eqn 5 (concat adjacent pairs).
T = 8
H = torch.randn(1, T, 4)                           # (N, T, d) toy lower-layer outputs

def pyramid_reshape(H):                            # Eqn 5: concat consecutive time-step pairs
    N, T, d = H.shape
    T = T - (T % 2)                                # drop a dangling odd frame
    H = H[:, :T, :].reshape(N, T // 2, 2 * d)      # two steps -> one wider step
    return H

len1 = pyramid_reshape(H).shape[1]
len2 = pyramid_reshape(pyramid_reshape(H)).shape[1]
len3 = pyramid_reshape(pyramid_reshape(pyramid_reshape(H))).shape[1]
print("pyramid lengths:", T, "->", len1, "->", len2, "->", len3)   # 8 -> 4 -> 2 -> 1

# Worked example (b): one content-attention step, Eqns 9-11 (phi=psi=identity -> dot).
h = torch.tensor([[2., 0.], [0., 2.], [1., 1.]])  # U=3 listener features h_u (2-dim)
s = torch.tensor([1., 0.5])                        # speller state s_i
e = h @ s                                          # Eqn 9  e_u = <phi(s), psi(h_u)> with identity MLPs
alpha = torch.softmax(e, dim=0)                    # Eqn 10
c = (alpha.unsqueeze(-1) * h).sum(0)              # Eqn 11  weighted sum
print("e     =", [round(x, 4) for x in e.tolist()])      # [2.0, 1.0, 1.5]
print("alpha =", [round(x, 4) for x in alpha.tolist()])  # [0.5065, 0.1863, 0.3072]
print("context c =", [round(x, 4) for x in c.tolist()])  # [1.3202, 0.6798]

### Step 2 — Build the pyramidal Listener and the attention module

Now we wrap those pieces into reusable modules. `pBLSTM` runs the Eqn-5 pair-concatenation, then a BiLSTM — that is one pyramid layer. The `Listener` stacks a bottom BiLSTM and three pyramid layers, reducing time by $8\times$ ($T \to T/8$); its ablation branch swaps in plain BiLSTMs that keep the full length.

`Attention` is the learnable form of worked example (b): small MLPs $\phi, \psi$ project the decoder state and encoder features into a shared key space before the dot-product score, softmax, and weighted sum.

In [ ]:
# The pyramidal BiLSTM Listener (built by hand). Eqn 5.
class pBLSTM(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.rnn = nn.LSTM(2 * in_dim, hid, batch_first=True, bidirectional=True)
    def forward(self, H):                          # H:(N,T,in_dim) -> (N,T/2,2*hid)
        N, Tlen, d = H.shape
        Tlen = Tlen - (Tlen % 2)
        H = H[:, :Tlen, :].reshape(N, Tlen // 2, 2 * d)   # Eqn 5: concat adjacent pairs
        return self.rnn(H)[0]

class Listener(nn.Module):
    def __init__(self, in_dim, hid, pyramid=True):
        super().__init__()
        self.pyramid = pyramid
        self.bottom = nn.LSTM(in_dim, hid, batch_first=True, bidirectional=True)
        if pyramid:
            self.p1 = pBLSTM(2 * hid, hid)
            self.p2 = pBLSTM(2 * hid, hid)
            self.p3 = pBLSTM(2 * hid, hid)
        else:   # ABLATION: same depth, NO downsampling (plain BiLSTMs)
            self.p1 = nn.LSTM(2 * hid, hid, batch_first=True, bidirectional=True)
            self.p2 = nn.LSTM(2 * hid, hid, batch_first=True, bidirectional=True)
            self.p3 = nn.LSTM(2 * hid, hid, batch_first=True, bidirectional=True)
    def forward(self, x):
        H = self.bottom(x)[0]
        if self.pyramid:
            return self.p3(self.p2(self.p1(H)))           # T -> T/2 -> T/4 -> T/8
        return self.p3(self.p2(self.p1(H)[0])[0])[0]      # T -> T (no reduction)

# Content-based attention (built by hand). Eqns 9-11.
class Attention(nn.Module):
    def __init__(self, s_dim, h_dim, key=64):
        super().__init__()
        self.phi = nn.Linear(s_dim, key)           # phi(s_i)
        self.psi = nn.Linear(h_dim, key)           # psi(h_u)
    def forward(self, s, H):                        # s:(N,s_dim)  H:(N,U,h_dim)
        e = (self.phi(s).unsqueeze(1) * self.psi(H)).sum(-1)   # Eqn 9 -> (N,U)
        alpha = torch.softmax(e, dim=1)            # Eqn 10  (over encoder steps)
        c = (alpha.unsqueeze(-1) * H).sum(1)       # Eqn 11  weighted sum -> (N,h_dim)
        return c, alpha

### Step 3 — Build the attention Speller (the decoder)

The `Speller` is a 2-layer LSTM decoder (Eqns 6–8). At every output step it embeds the previous character, concatenates the previous context, runs the two LSTM cells to get a fresh state $s_i$, then calls attention to read a **fresh context** $c_i$ from the encoder. The output character distribution is a linear layer over $[s_i; c_i]$.

We use teacher forcing — feeding the ground-truth previous character — and stash the attention weights each step so we can inspect the alignment later.

In [ ]:
# The attention Speller (2-layer LSTM decoder). Eqns 6-8.
VOC, IN, HID, EMB, OUTLEN = 8, 6, 32, 16, 6        # vocab, input dim, hidden, embed, output len

class Speller(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOC, EMB)
        self.attn = Attention(HID, 2 * HID)
        self.c1 = nn.LSTMCell(EMB + 2 * HID, HID)
        self.c2 = nn.LSTMCell(HID, HID)
        self.out = nn.Linear(HID + 2 * HID, VOC)   # CharacterDistribution on [s_i; c_i]
    def forward(self, H, tgt):                      # teacher forcing
        N = H.size(0)
        h1 = torch.zeros(N, HID); cstate1 = torch.zeros(N, HID)
        h2 = torch.zeros(N, HID); cstate2 = torch.zeros(N, HID)
        c = torch.zeros(N, 2 * HID)
        inp = torch.zeros(N, dtype=torch.long)      # BOS=0
        logits, attns = [], []
        for t in range(OUTLEN):
            cell1_in = torch.cat([self.emb(inp), c], -1)
            h1, cstate1 = self.c1(cell1_in, (h1, cstate1))    # Eqn 7
            h2, cstate2 = self.c2(h1, (h2, cstate2))
            c, a = self.attn(h2, H)                  # Eqn 6 / 9-11  fresh context every step
            logits.append(self.out(torch.cat([h2, c], -1)))   # Eqn 8
            attns.append(a)
            inp = tgt[:, t]
        return torch.stack(logits, 1), torch.stack(attns, 1)

### Step 4 — Train on a toy task and compare against the no-pyramid ablation

The toy task reads a 48-frame input and emits one character per 8-frame chunk (so the pyramid's $T/8$ reduction lines up with $\text{OUTLEN}=6$). We train the full Listener+Speller, then read the average attention map — a clean left-to-right diagonal means each output character learned to read its own chunk.

Finally we rerun with `pyramid=False`. Without downsampling the Speller must attend over $8\times$ more, nearly-identical steps, so it fits slower and its alignment is blurrier — reproducing the paper's claim that the pyramid is what makes attention converge.

In [ ]:
# Toy task: read a length-T input, emit a length-OUTLEN target (a downsampled "transcription").
def make(n):
    x = torch.randn(n, 48, IN)                     # T = 48 frames -> pyramid -> U = 6
    y = (x[:, ::8, 0] > 0).long()                  # target char per 8-frame chunk (toy label)
    return x, y

def run(pyramid, epochs=25, N=2000):
    torch.manual_seed(0)
    lis = Listener(IN, HID, pyramid=pyramid)
    spl = Speller()
    opt = torch.optim.Adam(list(lis.parameters()) + list(spl.parameters()), lr=3e-3)
    X, Y = make(N)
    for ep in range(epochs):
        perm = torch.randperm(N)
        for i in range(0, N, 128):
            b = perm[i:i + 128]
            logits, _ = spl(lis(X[b]), Y[b])
            loss = F.cross_entropy(logits.reshape(-1, VOC), Y[b].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
    Xt, Yt = make(400)
    with torch.no_grad():
        logits, attns = spl(lis(Xt), Yt)
        acc = (logits.argmax(-1) == Yt).float().mean().item()
        A = attns.mean(0)                          # avg alignment (OUTLEN x U)
    return acc, A

acc, A = run(pyramid=True)
print("\nPYRAMID Speller char accuracy:", round(acc, 4))
print("avg alignment (rows = output char, cols = encoder step):")
for row in A.tolist():
    print("  ", [round(x, 3) for x in row])

acc0, _ = run(pyramid=False)
print("\nNO-PYRAMID (ablation) char accuracy:", round(acc0, 4))
print("Pyramid fits faster with a crisp diagonal; no-pyramid attends over 8x more steps and is blurrier/slower.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported WER.)

## Visualize the data & results

_Does the pyramidal Listener + content-attention Speller learn a left-to-right reading alignment — and does removing the pyramid hurt?_

### Step 1 — Redefine the LAS modules standalone

This visualization block is self-contained, so we re-import torch and redefine compact versions of `pBLSTM`, `Listener`, `Attention`, and `Speller` (pyramid-only this time). The logic matches the reference implementation above; we restate it so this section runs on its own.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproduces the qualitative effect: pyramidal encoder + content attention -> diagonal reading alignment.
torch.manual_seed(0)
VOC, IN, HID, EMB, OUTLEN, N = 8, 6, 32, 16, 6, 2000

class pBLSTM(nn.Module):
    def __init__(self, d, hid):
        super().__init__()
        self.rnn = nn.LSTM(2 * d, hid, batch_first=True, bidirectional=True)
    def forward(self, H):
        N, T, d = H.shape
        T -= T % 2
        return self.rnn(H[:, :T, :].reshape(N, T // 2, 2 * d))[0]     # Eqn 5: concat adjacent pairs

class Listener(nn.Module):
    def __init__(self):
        super().__init__()
        self.bottom = nn.LSTM(IN, HID, batch_first=True, bidirectional=True)
        self.p1 = pBLSTM(2 * HID, HID)
        self.p2 = pBLSTM(2 * HID, HID)
        self.p3 = pBLSTM(2 * HID, HID)
    def forward(self, x):
        return self.p3(self.p2(self.p1(self.bottom(x)[0])))   # T -> T/8

class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.phi = nn.Linear(HID, 64)
        self.psi = nn.Linear(2 * HID, 64)
    def forward(self, s, H):
        e = (self.phi(s).unsqueeze(1) * self.psi(H)).sum(-1)     # Eqn 9
        a = torch.softmax(e, dim=1)                              # Eqn 10
        return (a.unsqueeze(-1) * H).sum(1), a                   # Eqn 11

class Speller(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOC, EMB)
        self.attn = Attention()
        self.c1 = nn.LSTMCell(EMB + 2 * HID, HID)
        self.c2 = nn.LSTMCell(HID, HID)
        self.out = nn.Linear(HID + 2 * HID, VOC)
    def forward(self, H, tgt):
        n = H.size(0)
        h1 = torch.zeros(n, HID); s1 = torch.zeros(n, HID)
        h2 = torch.zeros(n, HID); s2 = torch.zeros(n, HID)
        c = torch.zeros(n, 2 * HID)
        inp = torch.zeros(n, dtype=torch.long)
        lo, at = [], []
        for t in range(OUTLEN):
            h1, s1 = self.c1(torch.cat([self.emb(inp), c], -1), (h1, s1))   # Eqn 7
            h2, s2 = self.c2(h1, (h2, s2))
            c, a = self.attn(h2, H)                                          # Eqn 6/9-11
            lo.append(self.out(torch.cat([h2, c], -1)))                     # Eqn 8
            at.append(a)
            inp = tgt[:, t]
        return torch.stack(lo, 1), torch.stack(at, 1)

### Step 2 — Train the model on the toy reading task

We generate the same toy data (one character per 8-frame chunk) and train the Listener and Speller jointly for 25 epochs with Adam. This is the training loop that will produce the alignment we inspect next.

In [ ]:
def make(n):
    x = torch.randn(n, 48, IN)
    y = (x[:, ::8, 0] > 0).long()
    return x, y

lis = Listener()
spl = Speller()
opt = torch.optim.Adam(list(lis.parameters()) + list(spl.parameters()), lr=3e-3)
X, Y = make(N)
for ep in range(25):
    perm = torch.randperm(N)
    for i in range(0, N, 128):
        b = perm[i:i + 128]
        logits, _ = spl(lis(X[b]), Y[b])
        loss = F.cross_entropy(logits.reshape(-1, VOC), Y[b].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()

### Step 3 — Read out the learned alignment

On held-out data we measure character accuracy and average the attention weights into an OUTLEN-by-U map. A diagonal-dominant map — output char $i$ pointing at encoder step $i$ — is the signature of a working LAS: each emitted character reads essentially one audio chunk.

In [ ]:
Xt, Yt = make(400)
with torch.no_grad():
    logits, attns = spl(lis(Xt), Yt)
    acc = (logits.argmax(-1) == Yt).float().mean().item()
    A = attns.mean(0)
print("char accuracy:", round(acc, 4))
for row in A.tolist():
    print([round(x, 3) for x in row])
# Diagonal-dominant: output char i attends to encoder step i. Our run, not the paper's WER.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The pyramid ablation. You have a working LAS toy model with a 3-layer pyramidal
            Listener that converges fast and shows a clean diagonal alignment. Replace the pyramid with a
            plain BiLSTM that does no time downsampling (so the Speller attends over all $T$
            input steps). What do you expect to happen to convergence and to the alignment, and what paper
            claim does that reproduce?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap each pBLSTM for a same-width BiLSTM with the pair-concatenation removed, so the encoder length stays $T$ instead of $T/8$. — _An honest ablation changes exactly one thing &mdash; the time downsampling &mdash; so any difference is attributable to the pyramid._
- Retrain with everything else identical; track epochs-to-fit and per-character accuracy, and plot the alignment over the full-length encoder. — _With $8\times$ more nearly-identical encoder steps, the softmax has many similar candidates and struggles to sharpen onto the right chunk._
- Compare: the pyramid model fits faster with a crisp diagonal; the no-pyramid model fits slower (or worse) and its alignment is blurrier / more spread. — _Fewer, more distinct encoder chunks make the content match easier to localize &mdash; exactly the pyramid's purpose._

**Answer:** The no-pyramid model converges more slowly and tends to a blurrier alignment
                 (attention spread over many similar neighboring frames), often at lower accuracy. Since the
                 only change is removing the time downsampling, this isolates the pyramid as the cause &mdash;
                 reproducing the paper's claim that "without the pyramid structure in the encoder &hellip; our
                 model converges too slowly" (&sect;1). The pyramid gives attention a smaller set of distinct
                 chunks to point at.

</details>

**Problem 2.** Your worked example had $h_1=[2,0]$, $h_2=[0,2]$, $h_3=[1,1]$, $s=[1,0.5]$, giving
            $\alpha=[0.5065,0.1863,0.3072]$ and context $c=[1.320,0.680]$. Suppose training drives the
            energies so far apart that softmax returns $\alpha=[1,0,0]$. What is the context now, and what
            does that limiting case illustrate about the paper's remark that "$\alpha_i$ is typically very
            sharp"?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into Eqn 11: $c = 1\cdot[2,0] + 0\cdot[0,2] + 0\cdot[1,1] = [2,0]$. — _A one-hot $\alpha$ makes the weighted sum equal a single feature &mdash; here $h_1$._
- Note this equals "hard-select encoder step 1" &mdash; reading exactly one audio chunk. — _Soft attention contains hard selection as its sharp limit; the paper says convergence pushes $\alpha$ toward this._
- Contrast with the actual $\alpha=[0.5065,0.1863,0.3072]$: the real context $[1.320,0.680]$ blends all three, mostly $h_1$. — _Early in training $\alpha$ is soft (gradients flow); as it sharpens, the Speller reads one chunk per character, sweeping the diagonal of Fig. 2._

**Answer:** With $\alpha=[1,0,0]$ the context is $c=[2,0]=h_1$ &mdash; attention collapses to a
                 hard pick of encoder step 1. So the paper's "very sharp $\alpha$" is this
                 near-one-hot regime: at convergence each emitted character reads essentially one audio
                 chunk, which is exactly why the alignment in Fig. 2 looks like a thin diagonal band rather
                 than a smear. Softmax keeps it differentiable on the way there.

</details>

**Problem 3.** A classmate says: "Since LAS predicts each character $P(y_i\mid x, y_{<i})$ conditioned on
            previous characters, it must already contain a full language model, so language-model rescoring
            in the paper is pointless." Are they right?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note Eqn 1 conditions each character on $y_{<i}$, so LAS does learn spelling and short-range word structure &mdash; unlike CTC, which assumes characters are independent. — _This is the genuine advantage the paper claims over CTC; that part of the classmate's reasoning is correct._
- But observe LAS is trained only on the transcripts paired with audio &mdash; a limited text corpus &mdash; whereas an external language model is trained on far more text. — _The implicit LM inside LAS is only as good as its training transcripts; it has not seen most word sequences._
- Recall the paper's own numbers: 14.1% WER without the LM drops to 10.3% with LM rescoring. The improvement is real and large. — _If the internal LM were already complete, rescoring could not help by 3.8 absolute WER points._

**Answer:** Partly, but no. LAS's chain-rule conditioning (Eqn 1) gives it an implicit
                 language model &mdash; a real edge over CTC's independence assumption. But that implicit LM is
                 trained only on the audio transcripts, a small text source, so it is incomplete. The paper's
                 own results show rescoring with a larger external language model cuts WER from 14.1% to
                 10.3% &mdash; concrete evidence the internal LM is not sufficient on its own.

</details>